In [3]:
!pip install jiwer

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 13.6 MB/s  0:00:00

   ---------------------------------------- 0/3 [rapidfuzz]
   -------------------------- ------------- 2/3 [jiwer]
   ---------------------------------------- 3/3 [jiwer]



In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


CUDA available: True
CUDA device: NVIDIA GeForce RTX 4060 Laptop GPU


In [ ]:
# ============================================================
# PROCESS ONLY LABELLED IAM WORD IMAGES (FASTER & CLEANER)
# ============================================================

import os
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image
from jiwer import wer, cer
import torch
from tqdm import tqdm


# =====================
# PATHS
# =====================
root = "C:\\Users\\Harshada\\OneDrive\\Desktop\\Dunzo\\data\\iam_word_database1\\iam_words\\words"      # root IAM words folder
labels_file = "C:\\Users\\Harshada\\OneDrive\\Desktop\\Dunzo\\data\\iam_word_database1\\words_new.txt" # words.txt file


# =====================
# STEP 1 — LOAD LABELS
# =====================
labels = {}

with open(labels_file, "r") as f:
    for line in f:
        if line.startswith("#") or line.strip() == "":
            continue

        parts = line.strip().split(" ")
        word_id = parts[0]           # e.g. a01-000u-00-00
        transcription = parts[-1]    # last field is ground truth

        labels[word_id] = transcription

print("Total labels loaded:", len(labels))


# =====================
# STEP 2 — LOAD TROCR
# =====================
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")
model.eval()


# =====================
# STEP 3 — PROCESS ONLY LABELLED IMAGES
# =====================

truths = []
preds = []

for word_id in tqdm(labels.keys(), desc="Processing Labelled IAM Words"):

    # IAM words folder structure:
    # words/a01/a01-000u/a01-000u-00-00.png
    form = word_id[:7]              # a01-000u
    folder = word_id[:3]            # a01

    img_path = f"{root}/{folder}/{form}/{word_id}.png"

    # skip if image missing
    if not os.path.exists(img_path):
        continue

    # safe load
    try:
        image = Image.open(img_path)
    except:
        continue

    # convert grayscale / RGBA → RGB
    if image.mode != "RGB":
        image = image.convert("RGB")

    # skip tiny corrupted images
    if image.size[0] < 5 or image.size[1] < 5:
        continue

    # ground truth label
    gt = labels[word_id]

    # ---- Run TrOCR ----
    pixel_values = processor(images=image, return_tensors="pt").pixel_values
    generated_ids = model.generate(pixel_values)
    pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    truths.append(gt)
    preds.append(pred_text)



# =====================
# STEP 4 — FINAL METRICS
# =====================
print("\n========== FINAL EVALUATION ==========")
print("Total valid samples evaluated:", len(truths))
print("WER:", wer(truths, preds))
print("CER:", cer(truths, preds))
print("=========================================")


Total labels loaded: 44564


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:  84%|########4 | 1.12G/1.33G [00:00<?, ?B/s]

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-handwritten and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Processing Labelled IAM Words:   5%|▌         | 2293/44564 [18:07<2:27:41,  4.77it/s] 